# Poultry Feed Dataset Classification
## Machine Learning Model Training and Evaluation
This notebook trains a CNN model to classify poultry feed images into four categories: 0%, 25%, 50%, and 100%.

## Step 1: Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

# Scikit-learn for metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

ModuleNotFoundError: No module named 'numpy'

## Step 2: Set Up Data Paths and Parameters

In [ ]:
# Define dataset path
DATA_PATH = r'data'

# Image parameters
IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
CHANNELS = 3

# Training parameters
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

# Class labels
CLASSES = ['0_percent', '25_percent', '50_percent', '100_percent']
NUM_CLASSES = len(CLASSES)

print(f"Dataset Path: {DATA_PATH}")
print(f"Image Size: {IMG_SIZE}")
print(f"Number of Classes: {NUM_CLASSES}")
print(f"Classes: {CLASSES}")

## Step 3: Load and Explore the Dataset

In [ ]:
# Function to count images in each class
def count_images(data_path, classes):
    image_counts = {}
    for class_name in classes:
        class_path = os.path.join(data_path, class_name)
        if os.path.exists(class_path):
            images = [f for f in os.listdir(class_path) 
                     if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
            image_counts[class_name] = len(images)
        else:
            image_counts[class_name] = 0
    return image_counts

# Count images in each class
image_counts = count_images(DATA_PATH, CLASSES)

# Display results
print("Dataset Overview:")
print("-" * 50)
for class_name, count in image_counts.items():
    print(f"{class_name}: {count} images")
print("-" * 50)
print(f"Total Images: {sum(image_counts.values())}")

# Create a DataFrame for better visualization
df_counts = pd.DataFrame(list(image_counts.items()), columns=['Class', 'Count'])
print("\n", df_counts)

### Visualize Class Distribution

In [ ]:
# Visualize class distribution
plt.figure(figsize=(12, 5))

# Bar plot
plt.subplot(1, 2, 1)
plt.bar(image_counts.keys(), image_counts.values(), color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
plt.xlabel('Class', fontsize=12, fontweight='bold')
plt.ylabel('Number of Images', fontsize=12, fontweight='bold')
plt.title('Distribution of Images Across Classes', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Pie chart
plt.subplot(1, 2, 2)
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
plt.pie(image_counts.values(), labels=image_counts.keys(), autopct='%1.1f%%', 
        colors=colors, startangle=90)
plt.title('Class Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## Step 4: Visualize Sample Images from Each Class

In [ ]:
# Display sample images from each class
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold')

for idx, class_name in enumerate(CLASSES):
    class_path = os.path.join(DATA_PATH, class_name)
    if os.path.exists(class_path):
        images = [f for f in os.listdir(class_path) 
                 if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
        
        # Display first 5 images from each class
        for i in range(min(5, len(images))):
            img_path = os.path.join(class_path, images[i])
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[idx, i].imshow(img)
                axes[idx, i].axis('off')
                if i == 0:
                    axes[idx, i].set_title(f'{class_name}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## Step 5: Load and Preprocess Images

In [ ]:
# Function to load and preprocess images
def load_dataset(data_path, classes, img_size):
    images = []
    labels = []
    
    for class_idx, class_name in enumerate(classes):
        class_path = os.path.join(data_path, class_name)
        print(f"Loading images from {class_name}...")
        
        if os.path.exists(class_path):
            image_files = [f for f in os.listdir(class_path) 
                          if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
            
            for img_file in image_files:
                img_path = os.path.join(class_path, img_file)
                try:
                    # Read and preprocess image
                    img = cv2.imread(img_path)
                    if img is not None:
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        img = cv2.resize(img, img_size)
                        img = img / 255.0  # Normalize to [0, 1]
                        
                        images.append(img)
                        labels.append(class_idx)
                except Exception as e:
                    print(f"Error loading {img_path}: {e}")
        else:
            print(f"Warning: {class_path} does not exist!")
    
    return np.array(images), np.array(labels)

# Load the dataset
X, y = load_dataset(DATA_PATH, CLASSES, IMG_SIZE)

print(f"\nDataset loaded successfully!")
print(f"Images shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Image data type: {X.dtype}")
print(f"Label data type: {y.dtype}")
print(f"Pixel value range: [{X.min()}, {X.max()}]")

## Step 6: Split Dataset into Training and Validation Sets

In [ ]:
# Convert labels to categorical (one-hot encoding)
y_categorical = to_categorical(y, NUM_CLASSES)

# Split the dataset (80% training, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(
    X, y_categorical, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Dataset Split:")
print("-" * 50)
print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Training images shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Validation images shape: {X_val.shape}")
print(f"Validation labels shape: {y_val.shape}")

## Step 7: Create Data Augmentation Generator

In [ ]:
# Data augmentation for training set
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    shear_range=0.15,
    fill_mode='nearest'
)

# No augmentation for validation set
val_datagen = ImageDataGenerator()

print("Data augmentation configured successfully!")
print("\nAugmentation techniques applied:")
print("- Rotation: ±20 degrees")
print("- Width/Height shift: ±20%")
print("- Horizontal flip: Yes")
print("- Vertical flip: Yes")
print("- Zoom: ±20%")
print("- Shear: ±15%")

## Step 8: Build the CNN Model

In [ ]:
# Build CNN Model
model = Sequential([
    # First Convolutional Block
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),
    
    # Second Convolutional Block
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),
    
    # Third Convolutional Block
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),
    
    # Flatten and Dense Layers
    Flatten(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation='softmax')
])

# Display model architecture
model.summary()

### Visualize Model Architecture

In [ ]:
# Count total parameters
total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])

print(f"\nModel Statistics:")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

## Step 9: Compile the Model

In [ ]:
# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

print("Model compiled successfully!")
print(f"Optimizer: Adam (learning_rate={LEARNING_RATE})")
print(f"Loss Function: Categorical Crossentropy")
print(f"Metrics: Accuracy, Precision, Recall")

## Step 10: Set Up Callbacks and Train the Model

In [ ]:
# Define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    'best_poultry_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks = [early_stopping, reduce_lr, model_checkpoint]

print("Callbacks configured:")
print("- Early Stopping (patience=10)")
print("- Reduce Learning Rate on Plateau (patience=5)")
print("- Model Checkpoint (saves best model)")

### Train the Model

In [ ]:
# Train the model with data augmentation
print("Starting model training...")
print("=" * 70)

history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("=" * 70)
print("Training completed!")

## Step 11: Plot Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0, 0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0, 0].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].legend(loc='lower right')
axes[0, 0].grid(True, alpha=0.3)

# Loss
axes[0, 1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0, 1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 1].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Loss', fontsize=12)
axes[0, 1].legend(loc='upper right')
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(history.history['precision'], label='Training Precision', linewidth=2)
axes[1, 0].plot(history.history['val_precision'], label='Validation Precision', linewidth=2)
axes[1, 0].set_title('Model Precision Over Epochs', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Precision', fontsize=12)
axes[1, 0].legend(loc='lower right')
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history.history['recall'], label='Training Recall', linewidth=2)
axes[1, 1].plot(history.history['val_recall'], label='Validation Recall', linewidth=2)
axes[1, 1].set_title('Model Recall Over Epochs', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Recall', fontsize=12)
axes[1, 1].legend(loc='lower right')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 12: Evaluate Model Performance on Validation Set

In [ ]:
# Evaluate the model on validation set
print("Evaluating model on validation set...")
val_loss, val_accuracy, val_precision, val_recall = model.evaluate(X_val, y_val, verbose=0)

# Calculate F1-score
val_f1 = 2 * (val_precision * val_recall) / (val_precision + val_recall)

print("\n" + "=" * 70)
print("VALIDATION SET PERFORMANCE METRICS")
print("=" * 70)
print(f"Loss:      {val_loss:.4f}")
print(f"Accuracy:  {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
print(f"Precision: {val_precision:.4f}")
print(f"Recall:    {val_recall:.4f}")
print(f"F1-Score:  {val_f1:.4f}")
print("=" * 70)

### Visualize Performance Metrics

In [ ]:
# Visualize performance metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [val_accuracy, val_precision, val_recall, val_f1]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, values, color=['#4ECDC4', '#FF6B6B', '#45B7D1', '#96CEB4'], alpha=0.8, edgecolor='black')
plt.ylabel('Score', fontsize=12, fontweight='bold')
plt.title('Model Performance Metrics on Validation Set', fontsize=14, fontweight='bold')
plt.ylim([0, 1.1])
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.3f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## Step 13: Generate and Visualize Confusion Matrix

In [ ]:
# Make predictions on validation set
y_pred_prob = model.predict(X_val, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_val, axis=1)

# Generate confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASSES, yticklabels=CLASSES,
            cbar_kws={'label': 'Count'}, linewidths=1, linecolor='gray')
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Print confusion matrix
print("\nConfusion Matrix:")
print(cm)

### Normalized Confusion Matrix

In [ ]:
# Normalized confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='RdYlGn', 
            xticklabels=CLASSES, yticklabels=CLASSES,
            cbar_kws={'label': 'Percentage'}, linewidths=1, linecolor='gray')
plt.title('Normalized Confusion Matrix (Percentage)', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 14: Display Classification Report

In [ ]:
# Generate classification report
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
print("=" * 70)

# Create a DataFrame for better visualization
report_dict = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True)
df_report = pd.DataFrame(report_dict).transpose()
print("\nClassification Report (DataFrame):")
print(df_report)

### Visualize Per-Class Performance

In [ ]:
# Extract per-class metrics
class_metrics = {
    'Precision': [],
    'Recall': [],
    'F1-Score': []
}

for class_name in CLASSES:
    class_metrics['Precision'].append(report_dict[class_name]['precision'])
    class_metrics['Recall'].append(report_dict[class_name]['recall'])
    class_metrics['F1-Score'].append(report_dict[class_name]['f1-score'])

# Plot per-class performance
x = np.arange(len(CLASSES))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.bar(x - width, class_metrics['Precision'], width, label='Precision', color='#4ECDC4', alpha=0.8)
bars2 = ax.bar(x, class_metrics['Recall'], width, label='Recall', color='#FF6B6B', alpha=0.8)
bars3 = ax.bar(x + width, class_metrics['F1-Score'], width, label='F1-Score', color='#96CEB4', alpha=0.8)

ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASSES)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 1.1])

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Step 15: Visualize Predictions on Sample Images

In [ ]:
# Visualize predictions on random validation samples
num_samples = 16
indices = np.random.choice(len(X_val), num_samples, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('Sample Predictions on Validation Set', fontsize=16, fontweight='bold')

for idx, ax in enumerate(axes.flat):
    sample_idx = indices[idx]
    img = X_val[sample_idx]
    true_label = CLASSES[y_true[sample_idx]]
    pred_label = CLASSES[y_pred[sample_idx]]
    confidence = y_pred_prob[sample_idx][y_pred[sample_idx]] * 100
    
    ax.imshow(img)
    ax.axis('off')
    
    # Color code: green for correct, red for incorrect
    color = 'green' if true_label == pred_label else 'red'
    title = f'True: {true_label}\nPred: {pred_label}\nConf: {confidence:.1f}%'
    ax.set_title(title, fontsize=10, color=color, fontweight='bold')

plt.tight_layout()
plt.show()

### Prediction Confidence Distribution

In [ ]:
# Analyze prediction confidence
max_confidences = np.max(y_pred_prob, axis=1)

plt.figure(figsize=(14, 5))

# Histogram of confidence scores
plt.subplot(1, 2, 1)
plt.hist(max_confidences, bins=30, color='#4ECDC4', alpha=0.7, edgecolor='black')
plt.xlabel('Confidence Score', fontsize=12, fontweight='bold')
plt.ylabel('Frequency', fontsize=12, fontweight='bold')
plt.title('Distribution of Prediction Confidence', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

# Box plot of confidence by class
plt.subplot(1, 2, 2)
confidence_by_class = [max_confidences[y_true == i] for i in range(NUM_CLASSES)]
bp = plt.boxplot(confidence_by_class, labels=CLASSES, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
plt.ylabel('Confidence Score', fontsize=12, fontweight='bold')
plt.xlabel('Class', fontsize=12, fontweight='bold')
plt.title('Confidence Distribution by Class', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean Confidence: {np.mean(max_confidences):.4f}")
print(f"Median Confidence: {np.median(max_confidences):.4f}")
print(f"Min Confidence: {np.min(max_confidences):.4f}")
print(f"Max Confidence: {np.max(max_confidences):.4f}")

## Summary and Conclusion

In [ ]:
# Final Summary
print("=" * 70)
print("POULTRY FEED CLASSIFICATION - FINAL SUMMARY")
print("=" * 70)
print(f"\nModel Architecture: Convolutional Neural Network (CNN)")
print(f"Input Shape: {IMG_HEIGHT}x{IMG_WIDTH}x{CHANNELS}")
print(f"Number of Classes: {NUM_CLASSES}")
print(f"Classes: {', '.join(CLASSES)}")
print(f"\nTraining Samples: {len(X_train)}")
print(f"Validation Samples: {len(X_val)}")
print(f"Total Parameters: {total_params:,}")
print(f"\n" + "-" * 70)
print("PERFORMANCE METRICS (Validation Set)")
print("-" * 70)
print(f"Accuracy:  {val_accuracy*100:.2f}%")
print(f"Precision: {val_precision:.4f}")
print(f"Recall:    {val_recall:.4f}")
print(f"F1-Score:  {val_f1:.4f}")
print("-" * 70)
print(f"\nModel saved as: best_poultry_model.h5")
print("=" * 70)